# 22-Multi-Task Architectures

For 21 lessons, we have operated under a strict assumption: a Neural Network has a single goal. We trained an entire network to predict a house price (Regression). We trained a completely separate network to predict if a transaction was fraudulent (Classification).

But in a modern enterprise, deploying 50 different models for 50 different features is an infrastructural nightmare. Furthermore, these tasks are often highly correlated. If you are predicting the *Price* of a car, and the *Category* of a car, both tasks rely on the exact same underlying visual features (wheels, doors, brand logos).

Instead of training two separate brains, we can train one highly efficient brain with two mouths. This is the engineering discipline of **Multi-Task Learning (MTL)**.

Multi-Task Learning is the architectural design where a single neural network is trained to optimize multiple Loss Functions simultaneously. By forcing the network to solve two problems at once, it learns deeper, more generalized mathematical representations of the data.

Let's set up our PyTorch environment to build a branching neural network.

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch Multi-Task Architecture Environment Ready.")

✅ PyTorch Multi-Task Architecture Environment Ready.


# 1. Hard Parameter Sharing (The Backbone and the Heads)

The most common enterprise architecture for MTL is **Hard Parameter Sharing**.

Instead of a straight line from Input to Output, the network looks like a "Y" or a tree.

1. **The Shared Backbone (The Trunk)**: The first several hidden layers are shared across all tasks. When an image or data row enters the network, this backbone extracts universal, generalized features.
2. **The Task-Specific Heads (The Branches)**: After the backbone, the network physically splits into separate neural pathways. One pathway ends in a Sigmoid neuron (for Classification), and the other pathway ends in a Linear neuron (for Regression).

### The Enterprise Advantages

* **Memory and Inference Efficiency**: If you evaluate 5 tasks using 5 separate models, you must run 5 massive Forward Passes. With MTL, you run the heavy Backbone exactly once, and only run the tiny lightweight Heads 5 times.
* **Implicit Regularization**: If Task A starts overfitting and trying to memorize the noise, Task B's loss gradient will mathematically act as an anchor, pulling the shared backbone's weights back toward generalized reality. The tasks regularize each other!

# 2. The Mathematics of Combined Loss

In a standard network, you calculate one loss and run `.backward()`.
In an MTL network, you output multiple predictions, and you calculate multiple distinct losses.

Because PyTorch's Autograd engine requires a single scalar value to initiate the Chain Rule, we must mathematically combine the losses into a **Total Loss**. We do this using a weighted sum:


$$L_{total} = w_1 L_1 + w_2 L_2 + \dots + w_n L_n$$

### The Scale Paradox

This equation introduces a massive engineering trap.
Imagine Task 1 is predicting House Price (MSE). The loss might be $5,000,000$.
Imagine Task 2 is predicting if the house has a pool (BCE). The loss is between $0.0$ and $1.0$.

If you simply add them together ($5,000,000 + 0.5$), the gradient from the BCE loss is mathematically obliterated by the MSE loss. The network will dedicate 99.999% of its brainpower to predicting the price, and completely ignore the pool.
To fix this, you must engineer the loss weights ($w_i$) to physically scale the losses so their numerical magnitudes are roughly equal during training, or use advanced techniques like Dynamic Weight Averaging.

# 3. Architecting a Multi-Task Network in PyTorch

Let's build a complex Enterprise Neural Network. We will simulate an E-Commerce system that looks at a customer's profile (10 features) and simultaneously predicts two completely different things:

1. **Regression Task**: How much money will they spend? (Predicting a continuous value).
2. **Classification Task**: Will they return the item? (Predicting a 0 or 1).

In [4]:
# 1. Define the Multi-Task Architecture
class ECommerceMTL(nn.Module):
    def __init__(self):
        super().__init__()
        
        # --- THE SHARED BACKBONE ---
        # These layers learn universal patterns about the customer
        self.shared_backbone = nn.Sequential(
            nn.Linear(10, 64),
            nn.GELU(),
            nn.Linear(64, 32),
            nn.GELU()
        )
        
        # --- TASK HEAD 1: Spend Prediction (Regression) ---
        self.spend_head = nn.Sequential(
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Linear(16, 1) # Outputs a raw continuous number
        )
        
        # --- TASK HEAD 2: Return Prediction (Binary Classification) ---
        self.return_head = nn.Sequential(
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Linear(16, 1) # Outputs a raw logit (We use BCEWithLogitsLoss)
        )

    def forward(self, x):
        # 1. Pass data through the shared trunk
        shared_features = self.shared_backbone(x)
        
        # 2. Branch off into the individual task heads
        spend_pred = self.spend_head(shared_features)
        return_logit = self.return_head(shared_features)
        
        # Return both predictions simultaneously!
        return spend_pred, return_logit

# 2. Setup the Environment
torch.manual_seed(42)
model = ECommerceMTL()
optimizer = optim.AdamW(model.parameters(), lr=0.01)

# We need TWO separate Loss Functions!
criterion_spend = nn.MSELoss()
criterion_return = nn.BCEWithLogitsLoss()

# 3. Simulate Data
# 4 Customers, 10 Features
X_batch = torch.randn(4, 10) 
# Ground Truth Spend (e.g., $150, $20, $300, $0)
y_spend = torch.tensor([[150.0], [20.0], [300.0], [0.0]]) 
# Ground Truth Return (0=Keep, 1=Return)
y_return = torch.tensor([[0.0], [1.0], [1.0], [0.0]]) 

# 4. The Multi-Task Training Step
print("--- 🧠 Executing Multi-Task Forward Pass ---")
optimizer.zero_grad()

# The model yields two separate outputs
pred_spend, pred_return = model(X_batch)

print("Spend Predictions:\n", pred_spend.detach().numpy().round(1))
print("Return Logits:\n", pred_return.detach().numpy().round(2), "\n")

# Calculate the individual losses
loss_spend = criterion_spend(pred_spend, y_spend)
loss_return = criterion_return(pred_return, y_return)

print("--- ⚖️ Loss Calculation & Scaling ---")
print(f"Raw Spend Loss (MSE):  {loss_spend.item():.4f}")
print(f"Raw Return Loss (BCE): {loss_return.item():.4f}")

# CRITICAL MLOPS STEP: Scale the losses!
# Because MSE is in the tens-of-thousands, and BCE is 0.7, 
# we multiply BCE by a massive weight so it isn't ignored during Backprop.
weight_spend = 1.0
weight_return = 20000.0 

# The Combined Loss Equation
total_loss = (weight_spend * loss_spend) + (weight_return * loss_return)
print(f"Total Scaled Loss:     {total_loss.item():.4f}\n")

print("--- 📉 Executing Combined Backpropagation ---")
# This single backward pass calculates the Chain Rule down the branches 
# and safely merges the gradients through the shared trunk!
total_loss.backward()
optimizer.step()

print("✅ Multi-Task weights updated successfully.")

--- 🧠 Executing Multi-Task Forward Pass ---
Spend Predictions:
 [[0.2]
 [0.2]
 [0.2]
 [0.3]]
Return Logits:
 [[0.18]
 [0.18]
 [0.18]
 [0.18]] 

--- ⚖️ Loss Calculation & Scaling ---
Raw Spend Loss (MSE):  28168.2988
Raw Return Loss (BCE): 0.6985
Total Scaled Loss:     42137.5078

--- 📉 Executing Combined Backpropagation ---
✅ Multi-Task weights updated successfully.


## Real-World Use Case or Analogy:

Think of Multi-Task Architecture like **Tesla's Autopilot Vision System**:

* **The Problem**: A self-driving car processes 60 frames of video per second. For every frame, the car needs to know:
1. Where are the lane lines? (Segmentation)
2. How far away is the car ahead? (Depth Regression)
3. Is that a Stop Sign or a Speed Limit sign? (Classification)


* **The Inefficient Way (Separate Models)**: You train 3 completely different neural networks. You pass the heavy HD video frame through Network A, then Network B, then Network C. The hardware overheats, the latency drops to 5 frames per second, and the car crashes.
* **Multi-Task Learning (The Tesla Way)**: Tesla uses a massive Shared Backbone (a giant Convolutional Neural Network). This backbone looks at the video frame exactly *once* and learns fundamental physical features (edges, shapes, shadows). At the very end of the network, it branches off into dozens of tiny, lightweight "Heads." Head 1 takes those shapes and predicts depth; Head 2 takes those same shapes and classifies stop signs. The car achieves 60 frames per second using a fraction of the hardware, and the tasks make each other smarter!